# BÀI THỰC HÀNH SỐ 1 - KHAI PHÁ DỮ LIỆU
## PHẦN 2 - BÀI 2 (TRANG 37, 38): KHAI PHÁ LUẬT KẾT HỢP DỮ LIỆU ONLINE RETAIL
- **Họ và tên:** Bùi Minh Trọng
- **MSSV:** 6451071081
- **Lớp:** CQ.64.CNTT
- **Tập dữ liệu:** `Online Retail.xlsx`


### Câu 1: Đọc dữ liệu vào chương trình


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

data_candidates = ['Online Retail.xlsx', '../Online Retail.xlsx', '../../TH1/Online Retail.xlsx', '../TH1/Online Retail.xlsx']
data_path = next((p for p in data_candidates if os.path.exists(p)), 'Online Retail.xlsx')
df_retail = pd.read_excel(data_path)

print(f"Kích thước tập dữ liệu: {df_retail.shape[0]:,} dòng x {df_retail.shape[1]} cột")
display(df_retail.head())
print("\nThông tin cấu trúc tập dữ liệu:")
print(df_retail.info())


**Nhận xét Câu 1:**
- Tập dữ liệu bán lẻ trực tuyến có 541,909 dòng và 8 cột thuộc tính bao gồm: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country.


### Câu 2: Tiền xử lý dữ liệu


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

df_retail['Description'] = df_retail['Description'].astype(str).str.strip()
df_retail.dropna(subset=['InvoiceNo'], inplace=True)
df_retail['InvoiceNo'] = df_retail['InvoiceNo'].astype(str)

print(f"Số dòng sau khi xử lý missing InvoiceNo: {len(df_retail):,}")
display(df_retail.head())


**Nhận xét Câu 2:**
- Đã cắt bỏ khoảng trắng thừa ở cột Description, loại bỏ các dòng không có InvoiceNo và chuyển InvoiceNo sang kiểu chuỗi.


### Câu 3: Xóa bỏ các hóa đơn tín dụng (chứa chữ 'C')


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

df_clean_retail = df_retail[~df_retail['InvoiceNo'].str.contains('C')].copy()

print(f"Số dòng trước khi xóa hóa đơn 'C': {len(df_retail):,}")
print(f"Số dòng sau khi xóa hóa đơn 'C': {len(df_clean_retail):,}")
print(f"Số lượng hóa đơn tín dụng đã loại bỏ: {len(df_retail) - len(df_clean_retail):,}")


**Nhận xét Câu 3:**
- Đã loại bỏ các giao dịch trả hàng/hủy đơn (hóa đơn tín dụng có chữ 'C') để phản ánh chính xác hành vi mua hàng thực tế.


### Câu 4: Thống kê số dòng dữ liệu theo từng quốc gia


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

country_counts = df_clean_retail['Country'].value_counts()
print("Top 10 quốc gia có số lượng giao dịch nhiều nhất:")
display(country_counts.head(10))

plt.figure(figsize=(10, 4))
country_counts.head(10).plot(kind='bar', color='darkcyan')
plt.title("Top 10 Quốc gia có số lượng giao dịch nhiều nhất")
plt.xlabel("Quốc gia")
plt.ylabel("Số lượng giao dịch")
plt.xticks(rotation=45)
plt.show()


**Nhận xét Câu 4:**
- Nước Anh (United Kingdom) chiếm số lượng giao dịch áp đảo (gần 90% tổng số giao dịch). Theo sau là Đức (Germany) và Pháp (France).


### Câu 5: Lọc hóa đơn từ United Kingdom và gom nhóm Quantity


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

basket_uk = (df_clean_retail[df_clean_retail['Country'] == 'United Kingdom']
             .groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

print("Kích thước ma trận giỏ hàng của United Kingdom:", basket_uk.shape)
display(basket_uk.iloc[:5, :5])


**Nhận xét Câu 5:**
- Đã tạo thành công ma trận giỏ hàng cho thị trường Anh: mỗi hàng là một hóa đơn, mỗi cột là một mặt hàng, giá trị là tổng số lượng mua.


### Câu 6: Chuyển sang One-Hot, loại bỏ POSTAGE, tìm tập phổ biến và luật kết hợp


In [ ]:
print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

# Chuyển đổi sang One-Hot
basket_sets_uk = (basket_uk >= 1).astype(int)

# Xóa POSTAGE nếu có
if 'POSTAGE' in basket_sets_uk.columns:
    basket_sets_uk.drop('POSTAGE', inplace=True, axis=1)

from mlxtend.frequent_patterns import apriori, association_rules

frequent_itemsets = apriori(basket_sets_uk, min_support=0.03, use_colnames=True)
print(f"Số lượng tập phổ biến tìm được (min_sup = 3%): {len(frequent_itemsets)}")
display(frequent_itemsets.head(10))

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
print(f"\nSố lượng luật kết hợp tìm được (min_conf = 50%): {len(rules)}")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


**Nhận xét Câu 6:**
- Đã chuyển ma trận giỏ hàng sang dạng One-Hot 0/1 và loại bỏ cước phí POSTAGE.
- Thuật toán Apriori đã tìm được các tập mặt hàng phổ biến với độ hỗ trợ >= 3% và các luật kết hợp có độ tin cậy >= 50% mang ý nghĩa thực tiễn cao.
